## ERP000973

**paper:** [PMID: 22453055](https://pmc.ncbi.nlm.nih.gov/articles/PMC3328248/) - Annotation of primate miRNAs by high throughput sequencing of small RNA libraries, 2012

**date, curator:** 2026-03-30, Sara Carsanaro

**notes**
* per methods all animals are adults
* table 2 has male/female details, although was not able to confirm sex for orang brain due to inconsistent library naming

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,liver,UBERON:0002107,liver,perfect match
2,brain,UBERON:0000955,brain,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "ERP000973"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,,,,,,liver,adult,,,,M,,,9601,,,,,,ORANG2_liver,SAMEA1094503,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
1,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,,,,,,liver,adult,,,,M,,,9601,,,,,,ORANG1_liver,SAMEA1094504,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
2,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,,,,,,brain,adult,,,,NA,,,9601,,,,,,ORANG2_brain,SAMEA1094506,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
3,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,,,,,,brain,adult,,,,NA,,,9601,,,,,,ORANG1_brain,SAMEA1094502,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
4,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,,,,,,liver,adult,,,,F,,,9595,,,,,,GORILLA2_liver,SAMEA1094501,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
5,ERX027841,ERP000973,Illumina Genome Analyzer II,ERS068327,,,,,,liver,adult,,,,F,,,9595,,,,,,GORILLA1_liver,SAMEA1094508,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
6,ERX027840,ERP000973,Illumina Genome Analyzer,ERS068326,,,,,,brain,adult,,,,F,,,9595,,,,,,GORILLA2_brain,SAMEA1094505,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
7,ERX027839,ERP000973,Illumina Genome Analyzer,ERS068325,,,,,,brain,adult,,,,F,,,9595,,,,,,GORILLA1_brain,SAMEA1094507,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['brain' 'liver']


In [6]:

# brain
library.loc[library["infoOrgan"] == "brain", "anatId"] = "UBERON:0000955"
library.loc[library["infoOrgan"] == "brain", "anatName"] = "brain"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "brain", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "brain", "anatBiologicalStatus"] = "not documented"

# liver
library.loc[library["infoOrgan"] == "liver", "anatId"] = "UBERON:0002107"
library.loc[library["infoOrgan"] == "liver", "anatName"] = "liver"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "liver", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "liver", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,UBERON:0002107,liver,,,,liver,adult,perfect match,not documented,,M,,,9601,,,,,,ORANG2_liver,SAMEA1094503,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
1,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,UBERON:0002107,liver,,,,liver,adult,perfect match,not documented,,M,,,9601,,,,,,ORANG1_liver,SAMEA1094504,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
2,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,UBERON:0000955,brain,,,,brain,adult,perfect match,not documented,,NA,,,9601,,,,,,ORANG2_brain,SAMEA1094506,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
3,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,UBERON:0000955,brain,,,,brain,adult,perfect match,not documented,,NA,,,9601,,,,,,ORANG1_brain,SAMEA1094502,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
4,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,UBERON:0002107,liver,,,,liver,adult,perfect match,not documented,,F,,,9595,,,,,,GORILLA2_liver,SAMEA1094501,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
5,ERX027841,ERP000973,Illumina Genome Analyzer II,ERS068327,UBERON:0002107,liver,,,,liver,adult,perfect match,not documented,,F,,,9595,,,,,,GORILLA1_liver,SAMEA1094508,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
6,ERX027840,ERP000973,Illumina Genome Analyzer,ERS068326,UBERON:0000955,brain,,,,brain,adult,perfect match,not documented,,F,,,9595,,,,,,GORILLA2_brain,SAMEA1094505,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
7,ERX027839,ERP000973,Illumina Genome Analyzer,ERS068325,UBERON:0000955,brain,,,,brain,adult,perfect match,not documented,,F,,,9595,,,,,,GORILLA1_brain,SAMEA1094507,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,,,,,,ORANG2_liver,SAMEA1094503,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
1,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,,,,,,ORANG1_liver,SAMEA1094504,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
2,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,,,,,,ORANG2_brain,SAMEA1094506,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
3,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,,,,,,ORANG1_brain,SAMEA1094502,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
4,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,,,,,,GORILLA2_liver,SAMEA1094501,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
5,ERX027841,ERP000973,Illumina Genome Analyzer II,ERS068327,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,,,,,,GORILLA1_liver,SAMEA1094508,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
6,ERX027840,ERP000973,Illumina Genome Analyzer,ERS068326,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,,,,,,GORILLA2_brain,SAMEA1094505,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
7,ERX027839,ERP000973,Illumina Genome Analyzer,ERS068325,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,,,,,,GORILLA1_brain,SAMEA1094507,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'Small RNA Sample Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'miRNA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_liver,SAMEA1094503,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
1,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_liver,SAMEA1094504,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
2,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_brain,SAMEA1094506,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
3,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_brain,SAMEA1094502,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,ORANG1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
4,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_liver,SAMEA1094501,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
5,ERX027841,ERP000973,Illumina Genome Analyzer II,ERS068327,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA1_liver,SAMEA1094508,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
6,ERX027840,ERP000973,Illumina Genome Analyzer,ERS068326,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_brain,SAMEA1094505,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
7,ERX027839,ERP000973,Illumina Genome Analyzer,ERS068325,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA1_brain,SAMEA1094507,,,,,,,,,,,30/03/2026,Illumina SmallRNA protocol,,GORILLA1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation


#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-31'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_liver,SAMEA1094503,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,ORANG2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
1,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_liver,SAMEA1094504,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,ORANG1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
2,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_brain,SAMEA1094506,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,ORANG2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
3,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_brain,SAMEA1094502,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,ORANG1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
4,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_liver,SAMEA1094501,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,GORILLA2_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
5,ERX027841,ERP000973,Illumina Genome Analyzer II,ERS068327,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA1_liver,SAMEA1094508,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,GORILLA1_liver,,,,,public,TRANSCRIPTOMIC,size fractionation
6,ERX027840,ERP000973,Illumina Genome Analyzer,ERS068326,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_brain,SAMEA1094505,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,GORILLA2_brain,,,,,public,TRANSCRIPTOMIC,size fractionation
7,ERX027839,ERP000973,Illumina Genome Analyzer,ERS068325,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA1_brain,SAMEA1094507,,,,,,,,,,SAC,2026-03-31,Illumina SmallRNA protocol,,GORILLA1_brain,,,,,public,TRANSCRIPTOMIC,size fractionation


#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 22453055'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_liver,SAMEA1094503,,,,,,,PMID: 22453055,,,SAC,2026-03-31
1,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_liver,SAMEA1094504,,,,,,,PMID: 22453055,,,SAC,2026-03-31
2,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_brain,SAMEA1094506,,,,,,,PMID: 22453055,,,SAC,2026-03-31
3,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_brain,SAMEA1094502,,,,,,,PMID: 22453055,,,SAC,2026-03-31
4,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_liver,SAMEA1094501,,,,,,,PMID: 22453055,,,SAC,2026-03-31
5,ERX027841,ERP000973,Illumina Genome Analyzer II,ERS068327,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA1_liver,SAMEA1094508,,,,,,,PMID: 22453055,,,SAC,2026-03-31
6,ERX027840,ERP000973,Illumina Genome Analyzer,ERS068326,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_brain,SAMEA1094505,,,,,,,PMID: 22453055,,,SAC,2026-03-31
7,ERX027839,ERP000973,Illumina Genome Analyzer,ERS068325,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA1_brain,SAMEA1094507,,,,,,,PMID: 22453055,,,SAC,2026-03-31


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP000973,MicroRNA expression in primates II,"In addition to genome sequencing, accurate functional annotation of genomes is required in order to carry out comparative and evolutionary analyses between species. Among primates, the human genome is the most extensively annotated. Human miRNA gene annotation is based on multiple lines of evidence including evidence for expression as well as prediction of the characteristic hairpin structure. In contrast, most miRNA genes in non-human primates are annotated based on homology without any expression evidence. We have sequenced small-RNA libraries from chimpanzee, gorilla, orangutan and rhesus macaque from multiple individuals and tissues. Using patterns of miRNA expression in conjunction with a model of miRNA biogenesis we used these highthroughput sequencing data to identify novel miRNAs in non-human primates.",SRA,,,,,,,PRJEB2728,22453055,,10.1186/1471-2164-13-116,,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

8

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP000973,MicroRNA expression in primates II,"In addition to genome sequencing, accurate functional annotation of genomes is required in order to carry out comparative and evolutionary analyses between species. Among primates, the human genome is the most extensively annotated. Human miRNA gene annotation is based on multiple lines of evidence including evidence for expression as well as prediction of the characteristic hairpin structure. In contrast, most miRNA genes in non-human primates are annotated based on homology without any expression evidence. We have sequenced small-RNA libraries from chimpanzee, gorilla, orangutan and rhesus macaque from multiple individuals and tissues. Using patterns of miRNA expression in conjunction with a model of miRNA biogenesis we used these highthroughput sequencing data to identify novel miRNAs in non-human primates.",SRA,total,Bgee 1K,8,Small RNA Sample Prep Kit,full_length,,PRJEB2728,22453055,,10.1186/1471-2164-13-116,,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '22453055'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3328248/'
experiment.loc[:,'DOI'] = '10.1186/1471-2164-13-116'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP000973,MicroRNA expression in primates II,"In addition to genome sequencing, accurate functional annotation of genomes is required in order to carry out comparative and evolutionary analyses between species. Among primates, the human genome is the most extensively annotated. Human miRNA gene annotation is based on multiple lines of evidence including evidence for expression as well as prediction of the characteristic hairpin structure. In contrast, most miRNA genes in non-human primates are annotated based on homology without any expression evidence. We have sequenced small-RNA libraries from chimpanzee, gorilla, orangutan and rhesus macaque from multiple individuals and tissues. Using patterns of miRNA expression in conjunction with a model of miRNA biogenesis we used these highthroughput sequencing data to identify novel miRNAs in non-human primates.",SRA,total,Bgee 1K,8,Small RNA Sample Prep Kit,full_length,,PRJEB2728,22453055,https://pmc.ncbi.nlm.nih.gov/articles/PMC3328248/,10.1186/1471-2164-13-116,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
58022,SRX13395816,SRP350516,NextSeq 550,SRS11298977,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Brain,young adult,perfect match,not documented,perfect match,,,,60746,TruSeq Stranded Total RNA,full_length,ribo-minus,,,ASM1Brain,SAMN23970919,,,,,,,"PMID:35580607, Here we present comparative tra...",,,ANN,2026-03-31
58023,SRX13395815,SRP350516,NextSeq 550,SRS11298976,UBERON:0002048,lung,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Lung,young adult,perfect match,not documented,perfect match,,,,60746,TruSeq Stranded Total RNA,full_length,ribo-minus,,,AMS1Lung,SAMN23970876,,,,,,,"PMID:35580607, Here we present comparative tra...",,,ANN,2026-03-31
58024,ERX027846,ERP000973,Illumina Genome Analyzer,ERS068332,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_liver,SAMEA1094503,,,,,,,PMID: 22453055,,,SAC,2026-03-31
58025,ERX027845,ERP000973,Illumina Genome Analyzer II,ERS068331,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_liver,SAMEA1094504,,,,,,,PMID: 22453055,,,SAC,2026-03-31
58026,ERX027844,ERP000973,Illumina Genome Analyzer,ERS068330,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG2_brain,SAMEA1094506,,,,,,,PMID: 22453055,,,SAC,2026-03-31
58027,ERX027843,ERP000973,Illumina Genome Analyzer,ERS068329,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,brain,adult,perfect match,not documented,perfect match,NA,,,9601,Small RNA Sample Prep Kit,full_length,miRNA,,,ORANG1_brain,SAMEA1094502,,,,,,,PMID: 22453055,,,SAC,2026-03-31
58028,ERX027842,ERP000973,Illumina Genome Analyzer,ERS068328,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,9595,Small RNA Sample Prep Kit,full_length,miRNA,,,GORILLA2_liver,SAMEA1094501,,,,,,,PMID: 22453055,,,SAC,2026-03-31


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1136,SRP435772,Placental responses to chronic hypoxia and evo...,To understand how evolutionary adaptations hav...,SRA,partial,Bgee 1K,138,,,,PRJNA965800,37307471,https://pmc.ncbi.nlm.nih.gov/articles/PMC10288...,10.1073/pnas.2218049120,,"[Bgee curator notes: reproductive female, havi..."
1137,SRP350516,Comparative transcriptomics reveals circadian ...,Mammals differ more than 100-fold in maximum l...,SRA,total,Bgee 1K,522,TruSeq Stranded Total RNA,full_length,GSE190756,PRJNA788430,35580607,https://pmc.ncbi.nlm.nih.gov/articles/PMC9364679/,10.1016/j.cmet.2022.04.011,38052795[PMID],
1138,ERP000973,MicroRNA expression in primates II,"In addition to genome sequencing, accurate fun...",SRA,total,Bgee 1K,8,Small RNA Sample Prep Kit,full_length,,PRJEB2728,22453055,https://pmc.ncbi.nlm.nih.gov/articles/PMC3328248/,10.1186/1471-2164-13-116,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./
	../NCBI_output/

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop c4e26a1] adding annotated bulk experiment ERP000973
 2 files changed, 9 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.30 KiB | 2.30 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   16fc934..c4e26a1  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push